# 04 — Đánh giá trên Validation (S2-EVAL)

Gọi `src/evaluation_runner.evaluate_run` cho từng run và `comparison_table` để so sánh. Báo cáo hai cấp theo ADR-001:
**Macro F1 tổng thể 22 lớp** (chỉ số chính) và **Macro F1 nhóm 6 lớp liên quan nhựa** (`trash_bag`, `trash_bottle`,
`trash_snack_wrapper`, `trash_net`, `trash_rope`, `trash_tarp`). Phụ trách: Nguyễn Hoàng Ân (dự phòng: Yến).

**Sprint 2 chỉ dùng tập Validation.** Tập Test chạy một lần cho mô hình cuối ở Sprint 3, sau khi Yến xác nhận.

Kết quả: `results/metrics/<EXP_ID>_val.json`, `<EXP_ID>_val_predictions.csv`,
`results/figures/<EXP_ID>_val_confusion_matrix.png`, `<EXP_ID>_learning_curves.png`, `<EXP_ID>_val_misclassified.png`.

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token,
> không `git push`. Muốn sửa logic → sửa `src/` + `tests/` trên nhánh riêng và mở PR.
> Lưu notebook bằng *File → Save a copy in GitHub* vào **nhánh của mình**, xoá output trước khi commit.

## 1. Cài đặt và dữ liệu

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Dữ liệu đã xử lý (tạo bởi notebook 02). Colab: giải nén bản .zip trên Drive nếu chưa có.
from src.common import load_contract
DATA_VERSION = load_contract()["dataset"]["data_version"]      # v1_crop128
PROCESSED = WORK / "processed" / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"
if IN_COLAB and not PROCESSED.exists() and PROCESSED_ZIP.exists():
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)
assert (PROCESSED / "split_manifest.csv").exists(), f"Chưa có {PROCESSED} — chạy notebook 02 trước"
print("PROCESSED:", PROCESSED)

## 2. Chọn run cần đánh giá

In [ ]:
SPLIT = "val"
assert SPLIT == "val", "Sprint 2 không dùng tập Test"

RUNS = DRIVE / "runs"
RUN_DIRS = []   # ví dụ [RUNS / "E01_20261114-0930", RUNS / "E02_20261117-1015"]; để trống → lấy run mới nhất của mỗi EXP_ID
if not RUN_DIRS:
    latest = {}
    for d in sorted(RUNS.glob("E*_*")):
        if (d / "best.keras").exists():
            latest[d.name.split("_")[0]] = d
    RUN_DIRS = list(latest.values())
assert RUN_DIRS, f"Không có run nào có best.keras trong {RUNS} — chạy notebook 03 trước"
print("\n".join(map(str, RUN_DIRS)))

## 3. Đánh giá

In [ ]:
from src.evaluation_runner import evaluate_run, comparison_table

RESULTS = REPO / "results"
res = {}
for run in RUN_DIRS:
    r = evaluate_run(run, PROCESSED, split=SPLIT, results_dir=RESULTS)
    res[r["exp_id"]] = r
    g = r["plastic_related"]
    print(f"{r['exp_id']} | n={r['num_samples']} | Macro F1 22 lớp={r['overall']['macro_f1']:.4f}"
          f" | Macro F1 nhóm nhựa={g['macro_f1'] if g['macro_f1'] is None else round(g['macro_f1'], 4)}"
          f" | lớp nhựa không có mẫu: {g.get('classes_without_samples', [])}")

## 4. So sánh thí nghiệm

In [ ]:
from IPython.display import display
table = comparison_table(sorted((RESULTS / "metrics").glob(f"E*_{SPLIT}.json")))
table.to_csv(RESULTS / "metrics" / f"comparison_{SPLIT}.csv", index=False)
display(table)

## 5. Chỉ số theo lớp (kèm support — lớp support nhỏ không kết luận mạnh)

In [ ]:
import pandas as pd
for exp_id, r in res.items():
    per_class = pd.DataFrame(r["overall"]["per_class"]).T[["precision", "recall", "f1_score", "support"]]
    print(exp_id)
    display(per_class.sort_values("support", ascending=False).round(3))

## 6. Confusion matrix, learning curves, mẫu dự đoán sai

In [ ]:
from IPython.display import Image
for exp_id in res:
    for name in [f"{exp_id}_{SPLIT}_confusion_matrix.png", f"{exp_id}_learning_curves.png", f"{exp_id}_{SPLIT}_misclassified.png"]:
        p = RESULTS / "figures" / name
        if p.exists():
            print(name); display(Image(filename=str(p), width=900))

## Phụ lục — Smoke test `src/evaluate.py` (dữ liệu giả lập)

Kiểm tra công thức chỉ số trên mảng nhỏ có đáp án biết trước. **Đây là dữ liệu giả lập, không phải kết quả trên TrashCan.**

In [ ]:
from src.evaluate import smoke_test
assert smoke_test()